In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import csv
import json
import os
import glob

CSV_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv"
JSON_OUTPUT_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities"
EXTRACTED_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities"
OUTPUT_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs"

# Debug: show what files exist
print("Files in CSV_DIR:")
for f in sorted(glob.glob(os.path.join(CSV_DIR, "*.csv"))):
    print(f"  {os.path.basename(f)}")
print()

Files in CSV_DIR:
  all_entities.csv
  all_statements_with_attrs.csv
  faiksonmez_en_entities.csv
  faiksonmez_en_statements_with_attrs.csv
  gusto_en_entities.csv
  gusto_en_statements_with_attrs.csv
  hm_entities.csv
  hm_statements_with_attrs.csv
  mavi_en_entities.csv
  mavi_en_statements_with_attrs.csv



In [32]:
# Cell1: convert csv files to json
import csv
import json
import os
import glob

CSV_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/extracted_csv"
JSON_OUTPUT_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities"


# Get all company names from entity files
company_files = {}
for csv_file in sorted(glob.glob(os.path.join(CSV_DIR, "*_entities.csv"))):
    if "all_" in csv_file:
        continue
    safe_name = os.path.basename(csv_file).replace("_entities.csv", "")
    company_files[safe_name] = {"entities_csv": csv_file}

for csv_file in sorted(glob.glob(os.path.join(CSV_DIR, "*_statements_with_attrs.csv"))):
    if "all_" in csv_file:
        continue
    safe_name = os.path.basename(csv_file).replace("_statements_with_attrs.csv", "")
    if safe_name in company_files:
        company_files[safe_name]["statements_csv"] = csv_file
    else:
        company_files[safe_name] = {"statements_csv": csv_file}

# Build fresh JSON for each company
for safe_name, files in company_files.items():

    # --- Entities ---
    entities = []
    company_name = safe_name
    if "entities_csv" in files:
        with open(files["entities_csv"], "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                company_name = row["company"]
                entities.append({
                    "name": row["name"],
                    "type": row["type"],
                    "source_sentence": row["source_sentence"]
                })

# --- Statements ---
    statements = {}
    if "statements_csv" in files:
        with open(files["statements_csv"], "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                company_name = row["company"]
                stmt_text = row["statement"]

                # If statement is completely new, create it
                if stmt_text not in statements:
                    statements[stmt_text] = {
                        "statement": stmt_text,
                        "category": row["category"], # Start with the first category
                        "is_verifiable": row["is_verifiable"].lower() in ["true", "1", "yes"],
                        "subject": row["subject"] if row["subject"] else None,
                        "attributes": [],
                        "source_sentence": row["source_sentence"]
                    }
                else:
                    # If statement exists, check if this is a NEW category and append it
                    existing_categories = [c.strip() for c in statements[stmt_text]["category"].split(",")]
                    if row["category"] and row["category"] not in existing_categories:
                        statements[stmt_text]["category"] += f", {row['category']}"

                # Always append the attribute
                if row.get("attribute_name"):
                    statements[stmt_text]["attributes"].append({
                        "attribute_name": row["attribute_name"],
                        "attribute_value": row["attribute_value"],
                        "attribute_unit": row["attribute_unit"] if row["attribute_unit"] else None
                    })

    # --- Build fresh JSON ---
    output = {
        "company_name": company_name,
        "entities": entities,
        "factual_statements": list(statements.values()),
        "entity_count": len(entities),
        "statement_count": len(statements)
    }

    json_path = os.path.join(JSON_OUTPUT_DIR, f"{safe_name}_extracted.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"Created: {json_path} ({len(entities)} entities, {len(statements)} statements)")

print("\nAll JSON files rebuilt from CSV.")

Created: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/faiksonmez_en_extracted.json (11 entities, 15 statements)
Created: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/gusto_en_extracted.json (13 entities, 12 statements)
Created: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/hm_extracted.json (59 entities, 60 statements)
Created: /content/drive/MyDrive/06 - Green Washing AI/analysis/extracted_entities/mavi_en_extracted.json (101 entities, 76 statements)

All JSON files rebuilt from CSV.


In [33]:
# For KG construction, Load all individual company JSON files (skip the combined file)
company_data = {}
json_files = sorted(glob.glob(os.path.join(EXTRACTED_DIR, "*_extracted.json")))

for filepath in json_files:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    company_name = data.get("company_name", os.path.basename(filepath))
    company_data[company_name] = data
    e = data.get("entity_count", len(data.get("entities", [])))
    s = data.get("statement_count", len(data.get("factual_statements", [])))
    print(f"Loaded: {company_name} — {e} entities, {s} statements")

print(f"\nTotal companies loaded: {len(company_data)}")

Loaded: Faiksonmez En — 11 entities, 15 statements
Loaded: Gusto En — 13 entities, 12 statements
Loaded: Hm — 59 entities, 60 statements
Loaded: Mavi En — 101 entities, 76 statements

Total companies loaded: 4


In [34]:
# ============================================================
# CELL 3: Build separate knowledge graphs
# ============================================================

import networkx as nx
import re

def clean_name(name):
    """Normalize entity names for consistent matching."""
    name = name.strip()
    name = re.sub(r'\s+', ' ', name)
    return name

def find_related_entities(statement_text, entities_list):
    """
    Find which entities are mentioned in a factual statement.
    Returns a list of entity names found in the statement.
    """
    mentioned = []
    statement_lower = statement_text.lower()
    for entity in entities_list:
        entity_name = entity["name"]
        if len(entity_name) <= 3:
            continue
        if entity_name.lower() in statement_lower:
            mentioned.append(clean_name(entity_name))
    return mentioned


def build_company_graph(company_name, data):
    """
    Build a knowledge graph for a single company.
    Now includes structured attributes on nodes and edges.
    """
    G = nx.DiGraph()
    entities = data.get("entities", [])
    statements = data.get("factual_statements", [])

    # Determine the main company node name
    main_company = company_name
    for ent in entities:
        if ent["type"] == "Company":
            main_company = clean_name(ent["name"])
            break
    if main_company == company_name:
        for ent in entities:
            if ent["type"] == "Brand/Sub-brand":
                main_company = clean_name(ent["name"])
                break

    G.graph["main_company"] = main_company
    G.graph["company_key"] = company_name

    # --- Add entity nodes ---
    for entity in entities:
        node_name = clean_name(entity["name"])
        if G.has_node(node_name):
            existing_type = G.nodes[node_name].get("type", "")
            if entity["type"] not in existing_type:
                G.nodes[node_name]["type"] = existing_type + ", " + entity["type"] if existing_type else entity["type"]
        else:
            G.add_node(
                node_name,
                type=entity["type"],
                source_sentence=entity["source_sentence"],
                attributes={}  # Will store structured attributes
            )

    # --- Add factual statement edges with structured attributes ---
    for idx, stmt in enumerate(statements):
        statement_text = stmt["statement"]
        category = stmt["category"] # This now safely contains multiple categories like "Product_Info, Sustainability"
        is_verifiable = stmt.get("is_verifiable", False)
        source_sentence = stmt["source_sentence"]
        subject = stmt.get("subject", None)

        # Get the list of attributes from the JSON
        attributes_list = stmt.get("attributes", [])

        # If we have a subject and structured attribute, attach it to the subject node
        if subject and attributes_list:
            subject_clean = clean_name(subject)
            # Find the matching node
            matched_node = None
            for node in G.nodes():
                if node.lower() == subject_clean.lower() or subject_clean.lower() in node.lower() or node.lower() in subject_clean.lower():
                    matched_node = node
                    break

            if matched_node:
                # Attach ALL structured attributes to the node
                if "attributes" not in G.nodes[matched_node]:
                    G.nodes[matched_node]["attributes"] = {}
                for attr in attributes_list:
                    G.nodes[matched_node]["attributes"][attr["attribute_name"]] = {
                        "value": attr["attribute_value"],
                        "unit": attr.get("attribute_unit", None),
                        "verifiable": is_verifiable,
                        "source": statement_text
                    }

        # --- Edge creation ---
        mentioned_entities = find_related_entities(statement_text, entities)

        # Common edge attributes
        edge_attrs = {
            "relationship": "has_claim",
            "category": category,
            "is_verifiable": is_verifiable,
            "statement": statement_text
        }
        if attributes_list:
            edge_attrs["structured_attributes"] = attributes_list

        if len(mentioned_entities) == 0:
            stmt_node_id = f"STMT_{idx}"
            node_attrs = {
                "type": "FactualStatement",
                "statement": statement_text,
                "category": category,
                "is_verifiable": is_verifiable,
                "source_sentence": source_sentence
            }
            if attributes_list:
                node_attrs["structured_attributes"] = attributes_list
            G.add_node(stmt_node_id, **node_attrs)
            G.add_edge(main_company, stmt_node_id, **edge_attrs)

        elif len(mentioned_entities) == 1:
            target = mentioned_entities[0]
            if target == main_company:
                stmt_node_id = f"STMT_{idx}"
                node_attrs = {
                    "type": "FactualStatement",
                    "statement": statement_text,
                    "category": category,
                    "is_verifiable": is_verifiable,
                    "source_sentence": source_sentence
                }
                if attributes_list:
                    node_attrs["structured_attributes"] = attributes_list
                G.add_node(stmt_node_id, **node_attrs)
                G.add_edge(main_company, stmt_node_id, **edge_attrs)
            else:
                edge_attrs["relationship"] = "related_to"
                G.add_edge(main_company, target, **edge_attrs)

        else:
            for i in range(len(mentioned_entities)):
                for j in range(i + 1, len(mentioned_entities)):
                    ea = edge_attrs.copy()
                    ea["relationship"] = "related_to"
                    G.add_edge(mentioned_entities[i], mentioned_entities[j], **ea)
            for ent_name in mentioned_entities:
                if ent_name != main_company and not G.has_edge(main_company, ent_name):
                    ea = edge_attrs.copy()
                    ea["relationship"] = "mentions"
                    G.add_edge(main_company, ent_name, **ea)

    return G

# --- Build all graphs ---
company_graphs = {}

for company_name, data in company_data.items():
    G = build_company_graph(company_name, data)
    company_graphs[company_name] = G

    main = G.graph["main_company"]
    print(f"\n{'='*50}")
    print(f"Company: {company_name} (main node: {main})")
    print(f"  Nodes: {G.number_of_nodes()}")
    print(f"  Edges: {G.number_of_edges()}")

    # Node type breakdown
    # Node type breakdown
    print(f"  Node Types:")
    ntypes = {}
    for node, attrs in G.nodes(data=True):
        t = attrs.get("type", "Unknown")
        ntypes[t] = ntypes.get(t, 0) + 1
    for t, c in sorted(ntypes.items(), key=lambda x: -x[1]):
        print(f"    {t}: {c}")

    # Edge Category breakdown (This is where Sustainability will show up)
    print(f"  Edge Categories:")
    ecats = {}
    for u, v, attrs in G.edges(data=True):
        cats = attrs.get("category", "Unknown")
        # Because we merged categories earlier (e.g., "Product_Info, Sustainability"), we split them here to count them
        for c in cats.split(","):
            c = c.strip()
            ecats[c] = ecats.get(c, 0) + 1

    for c, count in sorted(ecats.items(), key=lambda x: -x[1]):
        print(f"    {c}: {count}")

    # Verifiable vs non-verifiable statements
    verifiable = sum(1 for _, _, a in G.edges(data=True) if a.get("is_verifiable", False))
    non_verifiable = G.number_of_edges() - verifiable
    print(f"  Verifiable edges: {verifiable}")
    print(f"  Non-verifiable edges: {non_verifiable}")

# Overall summary
print(f"\n{'='*50}")
print(f"ALL GRAPHS BUILT SUCCESSFULLY")
print(f"{'='*50}")
total_nodes = sum(G.number_of_nodes() for G in company_graphs.values())
total_edges = sum(G.number_of_edges() for G in company_graphs.values())
print(f"Total across all companies: {total_nodes} nodes, {total_edges} edges")


Company: Faiksonmez En (main node: Faik Sönmez)
  Nodes: 15
  Edges: 19
  Node Types:
    FactualStatement: 5
    Concept/Initiative: 4
    Product/Product_Line: 3
    Person, Company: 1
    Location: 1
    Target_Group: 1
  Edge Categories:
    Product_Info: 6
    Vague_Claim: 5
    History: 4
    Process: 3
    Metric: 2
  Verifiable edges: 15
  Non-verifiable edges: 4

Company: Gusto En (main node: Gusto)
  Nodes: 20
  Edges: 55
  Node Types:
    Location: 11
    FactualStatement: 7
    Company: 1
    Concept/Initiative: 1
  Edge Categories:
    Achievement: 45
    Vague_Claim: 5
    Geographic: 3
    Process: 2
    Organizational: 1
    Commitment/Goal: 1
    History: 1
  Verifiable edges: 2
  Non-verifiable edges: 53

Company: Hm (main node: H&M Group)
  Nodes: 76
  Edges: 115
  Node Types:
    Location: 17
    FactualStatement: 17
    Brand/Sub-brand: 14
    Concept/Initiative: 7
    Person: 6
    Organization: 5
    Company: 4
    Certification: 2
    Supplier: 1
    Event: 1
 

In [35]:
# ============================================================
# CELL 5: Export all graphs for later use in KG-RAG
# ============================================================

import pickle

for company_name, G in company_graphs.items():
    safe_name = company_name.lower().replace(" ", "_")

    # --- Pickle export ---
    pickle_path = os.path.join(OUTPUT_DIR, f"kg_{safe_name}.gpickle")
    with open(pickle_path, "wb") as f:
        pickle.dump(G, f)

    # --- JSON export ---
    graph_json = {
        "company_name": company_name,
        "main_company": G.graph.get("main_company", company_name),
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "nodes": [],
        "edges": []
    }
    for node, attrs in G.nodes(data=True):
        node_data = {"id": node}
        node_data.update(attrs)
        graph_json["nodes"].append(node_data)
    for u, v, attrs in G.edges(data=True):
        edge_data = {"source": u, "target": v}
        edge_data.update(attrs)
        graph_json["edges"].append(edge_data)

    json_path = os.path.join(OUTPUT_DIR, f"kg_{safe_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(graph_json, f, indent=2, ensure_ascii=False)

    print(f"Exported {company_name}: {pickle_path} + {json_path}")

# --- Combined stats file ---
all_stats = {}
for company_name, G in company_graphs.items():
    ntypes = {}
    for node, attrs in G.nodes(data=True):
        t = attrs.get("type", "Unknown")
        ntypes[t] = ntypes.get(t, 0) + 1
    ecats = {}
    for u, v, attrs in G.edges(data=True):
        c = attrs.get("category", "Unknown")
        ecats[c] = ecats.get(c, 0) + 1

    all_stats[company_name] = {
        "main_company": G.graph.get("main_company", company_name),
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "node_types": ntypes,
        "edge_categories": ecats,
        "verifiable_edges": sum(1 for _, _, a in G.edges(data=True) if a.get("is_verifiable", False)),
        "non_verifiable_edges": sum(1 for _, _, a in G.edges(data=True) if not a.get("is_verifiable", False))
    }

stats_path = os.path.join(OUTPUT_DIR, "all_graphs_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(all_stats, f, indent=2, ensure_ascii=False)

print(f"\nAll stats saved: {stats_path}")
print(f"\n{'='*50}")
print("STEP 3 COMPLETE — SEPARATE GRAPHS PER COMPANY")
print(f"{'='*50}")
for company_name, G in company_graphs.items():
    print(f"  {G.graph['main_company']}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"\nAll files saved to: {OUTPUT_DIR}")
print(f"\nNext step: Use each company's graph as the retrieval source")
print(f"for KG-RAG to generate and test sustainability marketing content.")

Exported Faiksonmez En: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_faiksonmez_en.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_faiksonmez_en.json
Exported Gusto En: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_gusto_en.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_gusto_en.json
Exported Hm: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_hm.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_hm.json
Exported Mavi En: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_mavi_en.gpickle + /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/kg_mavi_en.json

All stats saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs/all_graphs_stats.json

STEP 3 COMPLETE — SEPARATE GRAPHS PER COMPANY
  Faik Sönmez: 15 nodes, 19 edges
  Gusto: 20 no

In [36]:
!pip install pyvis

In [37]:
# ============================================================
# Interactive PyVis Visualizations Per Company KG
# ============================================================

from pyvis.network import Network
import os
import pickle
import glob
from pathlib import Path

# --- Paths ---
KG_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/knowledge_graphs"
VIS_DIR = "/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations"
os.makedirs(VIS_DIR, exist_ok=True)

# --- Color mapping ---
COLOR_MAP = {
    "Company": "#FF6B6B",
    "Brand/Sub-brand": "#FF8E8E",
    "Person": "#4ECDC4",
    "Location": "#45B7D1",
    "Certification": "#96CEB4",
    "Organization": "#FFEAA7",
    "Material": "#DDA0DD",
    "Product/Product_Line": "#98D8C8",
    "Partnership": "#F7DC6F",
    "Technology": "#BB8FCE",
    "Event": "#85C1E9",
    "Supplier": "#F1948A",
    "Concept/Initiative": "#73C6B6",
    "FactualStatement": "#D5D8DC",
    "Sustainability" : "#2ECC71",
    "Award" : "#E74C3C",
    "Policy": "#F39C12",
    "Commitment/Goal" : "#3498DB",
    "Other": "#AEB6BF",
}

def get_node_color(node_type):
    for key, color in COLOR_MAP.items():
        if key in str(node_type):
            return color
    return "#AEB6BF"

# --- Load and visualize each company KG ---
for pkl_file in sorted(glob.glob(os.path.join(KG_DIR, "*.gpickle"))):
    with open(pkl_file, "rb") as f:
        G = pickle.load(f)

    company_key = G.graph.get("company_key", Path(pkl_file).stem)
    main_company = G.graph.get("main_company", company_key)

    print(f"\nBuilding visualization for: {main_company} ({G.number_of_nodes()} nodes, {G.number_of_edges()} edges)")

    net = Network(
        height="800px",
        width="100%",
        bgcolor="#ffffff",
        font_color="#333333",
        notebook=True,
        cdn_resources="remote"
    )

    # --- Add nodes ---
    for node, attrs in G.nodes(data=True):
        node_type = attrs.get("type", "Other")
        color = get_node_color(node_type)

        if str(node).startswith("STMT_"):
            # Statement nodes — show shortened text
            statement = attrs.get("statement", str(node))
            label = statement[:40] + "..." if len(statement) > 40 else statement

            # Build tooltip with full details
            title_parts = [
                f"Type: FactualStatement",
                f"Category: {attrs.get('category', 'N/A')}",
                f"Verifiable: {attrs.get('is_verifiable', 'N/A')}",
                f"",
                f"Full statement: {statement}"
            ]

            # Loop through the list of attributes if present
            sa_list = attrs.get("structured_attributes", [])
            for sa in sa_list:
                title_parts.append(f"")
                title_parts.append(f"Attribute: {sa.get('attribute_name', '')}")
                title_parts.append(f"Value: {sa.get('attribute_value', '')} {sa.get('attribute_unit', '') or ''}")

            title = "\n".join(title_parts)
            size = 12

        else:
            # Entity nodes
            label = str(node)
            title_parts = [
                f"Type: {node_type}",
                f"Source: {attrs.get('source_sentence', 'N/A')}"
            ]
            # Show structured attributes stored on node
            node_attrs = attrs.get("attributes", {})
            if node_attrs:
                title_parts.append("")
                title_parts.append("Structured Attributes:")
                for attr_name, attr_data in node_attrs.items():
                    val = attr_data.get("value", "?")
                    unit = attr_data.get("unit", "")
                    unit_str = f" {unit}" if unit else ""
                    ver = "✓" if attr_data.get("verifiable", False) else "?"
                    title_parts.append(f"  {attr_name}: {val}{unit_str} [{ver}]")

            title = "\n".join(title_parts)

            # Larger size for main company node
            if node_type in ["Company", "Brand/Sub-brand"]:
                size = 35
            elif node_type in ["Certification", "Concept/Initiative"]:
                size = 25
            else:
                size = 18

        net.add_node(node, label=label, title=title, color=color, size=size)

    # --- Add edges ---
    for u, v, attrs in G.edges(data=True):
        relationship = attrs.get("relationship", "related_to")
        category = attrs.get("category", "")
        statement = attrs.get("statement", "")
        is_verifiable = attrs.get("is_verifiable", False)
        ver_tag = "✓ Verified" if is_verifiable else "? Unverified"

        title_parts = [
            f"Relationship: {relationship}",
            f"Category: {category}",
            f"Status: {ver_tag}",
            f"",
            f"Statement: {statement}"
        ]

        sa_list = attrs.get("structured_attributes", [])
        for sa in sa_list:
            title_parts.append(f"")
            title_parts.append(f"Attribute: {sa.get('attribute_name', '')} = {sa.get('attribute_value', '')} {sa.get('attribute_unit', '') or ''}")

        title = "\n".join(title_parts)

        # Color edges by verifiability
        edge_color = "#2ECC71" if is_verifiable else "#E74C3C"

        net.add_edge(u, v, title=title, color=edge_color, width=1.5)
    # --- Physics settings for better layout ---
    net.set_options("""
    {
        "physics": {
            "forceAtlas2Based": {
                "gravitationalConstant": -100,
                "centralGravity": 0.01,
                "springLength": 200,
                "springConstant": 0.02
            },
            "solver": "forceAtlas2Based",
            "stabilization": {
                "iterations": 200
            }
        },
        "interaction": {
            "hover": true,
            "tooltipDelay": 100,
            "navigationButtons": true
        }
    }
    """)

    # --- Save ---
    html_path = os.path.join(VIS_DIR, f"kg_{company_key.lower().replace(' ', '_')}_interactive.html")
    net.show(html_path)
    print(f"  Saved: {html_path}")

print(f"\n{'='*50}")
print(f"All interactive visualizations saved to: {VIS_DIR}")
print(f"Open the HTML files in your browser to explore.")
print(f"  Green edges = verified facts")
print(f"  Red edges = unverified claims")
print(f"  Hover over nodes/edges to see details and attributes")


Building visualization for: Faik Sönmez (15 nodes, 19 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_faiksonmez_en_interactive.html
  Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_faiksonmez_en_interactive.html

Building visualization for: Gusto (20 nodes, 55 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_gusto_en_interactive.html
  Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_gusto_en_interactive.html

Building visualization for: H&M Group (76 nodes, 115 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_hm_interactive.html
  Saved: /content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_hm_interactive.html

Building visualization for: Mavi (116 nodes, 154 edges)
/content/drive/MyDrive/06 - Green Washing AI/analysis/kg_visualizations/kg_mavi_en_interactive.html
  Saved: /content/drive/MyDrive/0